In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors

import joblib

# Load dataset

data = pd.read_csv("web_attacks_balanced.csv")

# Target column

target_column = "Label"

# Selected features

selected_features = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets'
]

# Features and target

X = data[selected_features]
y = data[target_column]

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Apply a SMOTE-style oversampler without requiring imblearn

def smote_resample(X, y, random_state=42, k_neighbors=5):
    rng = np.random.default_rng(random_state)
    X_df = pd.DataFrame(X).reset_index(drop=True)
    y_series = pd.Series(y).reset_index(drop=True)
    counts = y_series.value_counts()
    target_size = counts.max()
    samples = [X_df]
    labels = [y_series]

    for label, count in counts.items():
        if count == target_size:
            continue

        class_mask = y_series == label
        class_X = X_df.loc[class_mask].to_numpy()
        if len(class_X) < 2:
            extra = X_df.loc[class_mask].sample(target_size - count, replace=True, random_state=random_state)
            samples.append(extra)
            labels.append(pd.Series([label] * len(extra)))
            continue

        n_neighbors = min(k_neighbors, len(class_X) - 1)
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
        nn.fit(class_X)
        needed = target_size - count
        synthetic = []

        for _ in range(needed):
            idx = rng.integers(0, len(class_X))
            sample = class_X[idx]
            neighbors = nn.kneighbors([sample], return_distance=False)[0][1:]
            neighbor = class_X[rng.choice(neighbors)]
            gap = rng.random()
            synthetic.append(sample + gap * (neighbor - sample))

        samples.append(pd.DataFrame(synthetic, columns=X_df.columns))
        labels.append(pd.Series([label] * needed))

    X_resampled = pd.concat(samples, ignore_index=True)
    y_resampled = pd.concat(labels, ignore_index=True)
    return X_resampled, y_resampled

X_train_smote, y_train_smote = smote_resample(
    X_train,
    y_train,
    random_state=42
)

# Train model

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_smote, y_train_smote)

# Save model

joblib.dump(
    model,
    "random_forest_model_4_features.joblib"
)

print("Model saved successfully")

# Predictions

y_pred = model.predict(X_test)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)

# Classification Report

print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))


Model saved successfully

Accuracy: 0.8658872077028886

Classification Report:

                            precision    recall  f1-score   support

                    BENIGN       0.99      0.98      0.99      1035
  Web Attack – Brute Force       0.71      0.60      0.65       290
Web Attack – Sql Injection       0.33      0.20      0.25         5
          Web Attack – XSS       0.38      0.55      0.45       124

                  accuracy                           0.87      1454
                 macro avg       0.60      0.58      0.58      1454
              weighted avg       0.88      0.87      0.87      1454

